# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import os
import tempfile
import requests
from langchain_community.document_loaders import PyPDFLoader

PDF_URL = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

docs = []
pdf_path = None

def download_pdf_to_temp(url: str) -> str:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    tmpf = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
    try:
        tmpf.write(resp.content)
        tmpf.flush()
        return tmpf.name
    finally:
        tmpf.close()

try:
    pdf_path = download_pdf_to_temp(PDF_URL)
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print("Loaded PDF from URL.")
except Exception as e:
    raise RuntimeError(f"Could not load PDF from URL: {PDF_URL}. Error: {e}") from e
finally:
    if pdf_path and os.path.exists(pdf_path):
        try:
            os.remove(pdf_path)
        except OSError:
            pass

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Document loaded successfully! Total length: {len(document_text)} characters")
print("\nFirst 200 characters of the document:")
print(document_text[:200] + "...")

Loaded PDF from URL.
Document loaded successfully! Total length: 51452 characters

First 200 characters of the document:
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Pra...


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from pydantic import BaseModel
from openai import OpenAI
import json
import tiktoken
import os
import re

class ArticleAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

def get_encoding_for_model(model: str):
    try:
        return tiktoken.encoding_for_model(model)
    except Exception:
        return tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    enc = get_encoding_for_model(model)
    return len(enc.encode(text))

def truncate_to_token_limit(text: str, max_tokens: int = 6000, model: str = "gpt-4o") -> str:
    enc = get_encoding_for_model(model)
    toks = enc.encode(text)
    if len(toks) <= max_tokens:
        return text
    return enc.decode(toks[:max_tokens])

def trim_to_token_limit_nicely(text: str, max_tokens: int, model: str = "gpt-4o") -> str:
    enc = get_encoding_for_model(model)
    toks = enc.encode(text)
    if len(toks) <= max_tokens:
        return text
    trimmed = enc.decode(toks[:max_tokens])
    last_period = max(trimmed.rfind(". "), trimmed.rfind("! "), trimmed.rfind("? "))
    if last_period != -1 and (len(trimmed) - last_period) < 300:
        return trimmed[: last_period + 1]
    return trimmed

def enforce_single_paragraph(text: str, max_sentences: int = 4) -> str:
    one_para = " ".join(text.splitlines())
    sents = re.split(r'(?<=[.!?])\s+', one_para.strip())
    if len(sents) > max_sentences:
        one_para = " ".join(sents[:max_sentences]).strip()
    return one_para

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable is not set!")
client = OpenAI(api_key=api_key)

tone = "Bureaucratese"
model_name = "gpt-4o"
truncated_text = truncate_to_token_limit(document_text, max_tokens=6000, model=model_name)

system_prompt = (
    'You are an expert analyst tasked with analyzing and summarizing documents in a distinct style.\n'
    'Return ONLY valid JSON with these fields:\n'
    '- Author (string)\n'
    '- Title (string)\n'
    '- Relevance (one paragraph explaining why this is relevant for AI professionals)\n'
    '- Summary (concise, <= 1000 tokens)\n'
    f'- Tone (use exactly: "{tone}")\n\n'
    f'The summary must be written in the "{tone}" style with clear markers of that style.'
)

user_prompt = (
    'Please analyze the following document and return the analysis as a JSON object:\n\n'
    + truncated_text
)

response = client.chat.completions.create(
    model=model_name,
    response_format={"type": "json_object"},
    temperature=0,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)

content = response.choices[0].message.content
data = json.loads(content)
data["Tone"] = tone

# Enforce Relevance to one concise paragraph
if isinstance(data.get("Relevance"), str):
    data["Relevance"] = enforce_single_paragraph(data["Relevance"], max_sentences=4)

# Enforce Summary ≤ 1000 tokens post-generation
if isinstance(data.get("Summary"), str):
    data["Summary"] = trim_to_token_limit_nicely(data["Summary"], max_tokens=1000, model=model_name)

data["InputTokens"] = getattr(response.usage, "prompt_tokens", 0) or 0
data["OutputTokens"] = getattr(response.usage, "completion_tokens", 0) or 0

analysis = ArticleAnalysis.model_validate(data)

print("=== Article Analysis ===\n")
print(analysis.model_dump_json(indent=2))


=== Article Analysis ===

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This document is of paramount importance for AI professionals as it underscores the necessity of self-management in an era where traditional career paths are becoming obsolete. AI professionals, often working in dynamic and rapidly evolving environments, must cultivate a profound understanding of their strengths, values, and work styles to navigate their careers effectively. The insights provided by Drucker on self-assessment and personal development are crucial for AI professionals who must continuously adapt and contribute meaningfully in their field.",
  "Summary": "In the esteemed publication 'Managing Oneself' by Peter F. Drucker, the author elucidates the imperative of self-management in the contemporary knowledge economy. The document articulates that individuals must assume the role of their own chief executive officers, necessitating a comprehensive understanding of perso

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [4]:
from pydantic import BaseModel
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

summarization_questions = [
    "Does the summary capture the core thesis?",
    "Are the main supporting points represented accurately?",
    "Is the summary concise and within the expected scope?",
    "Does the summary preserve the original intent and nuance?",
    "Are crucial examples/practices (e.g., feedback analysis, values alignment) mentioned appropriately?",
]

coherence_questions = [
    "Is the structure logically ordered?",
    "Are transitions between ideas clear?",
    "Is the summary easy to follow?",
    "Is terminology used consistently?",
    "Are there no contradictions or abrupt topic shifts?",
]

tonality_questions = [
    "Does the writing adhere to the requested style?",
    "Is the voice consistently formal and institutional?",
    "Is passive/impersonal voice used appropriately?",
    "Is vocabulary aligned with the requested tone?",
    "Does the tone remain consistent throughout?",
]

safety_questions = [
    "Is the content free from harmful or biased statements?",
    "Does it avoid sensitive, controversial claims?",
    "Is the language neutral and objective?",
    "Are no stereotypes or derogatory terms used?",
    "Does the content remain within safe guidelines?",
]

summarization_metric = SummarizationMetric(
    model=model_name,
    assessment_questions=summarization_questions,
    threshold=0.0,
)
coherence_metric = GEval(
    name="Coherence",
    criteria=None,
    evaluation_steps=coherence_questions,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model_name,
    threshold=0.0,
    strict_mode=False,
)
tonality_metric = GEval(
    name="Tonality",
    criteria=None,
    evaluation_steps=tonality_questions,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model_name,
    threshold=0.0,
    strict_mode=False,
)
safety_metric = GEval(
    name="Safety",
    criteria=None,
    evaluation_steps=safety_questions,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model_name,
    threshold=0.0,
    strict_mode=False,
)

_test_case = LLMTestCase(input=document_text, actual_output=analysis.Summary)

summarization_metric.measure(_test_case)
coherence_metric.measure(_test_case)
tonality_metric.measure(_test_case)
safety_metric.measure(_test_case)

evaluation_results = EvaluationResults(
    SummarizationScore=float(summarization_metric.score or 0.0),
    SummarizationReason=str(summarization_metric.reason or ""),
    CoherenceScore=float(coherence_metric.score or 0.0),
    CoherenceReason=str(coherence_metric.reason or ""),
    TonalityScore=float(tonality_metric.score or 0.0),
    TonalityReason=str(tonality_metric.reason or ""),
    SafetyScore=float(safety_metric.score or 0.0),
    SafetyReason=str(safety_metric.reason or ""),
)

print("=== Evaluation Results ===\n")
print(evaluation_results.model_dump_json(indent=2))


Output()

Output()

Output()

Output()

=== Evaluation Results ===

{
  "SummarizationScore": 0.7272727272727273,
  "SummarizationReason": "The score is 0.73 because the summary contains a contradiction regarding the purpose of feedback analysis, which affects the accuracy. Additionally, it includes extra information about adapting work habits and thriving in specific roles, which were not present in the original text. Despite these issues, the summary still captures a significant portion of the original content effectively.",
  "CoherenceScore": 0.9119202917328538,
  "CoherenceReason": "The response is logically ordered, starting with the main theme of self-management and progressing through specific strategies like feedback analysis and understanding learning styles. Transitions between ideas are clear, with each concept building on the previous one. The summary is easy to follow, consistently using terminology related to self-management and personal development. There are no contradictions or abrupt topic shifts, maintain

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
import json


def build_enhancement_prompt(original_summary: str, eval_results: EvaluationResults, tone: str = "Bureaucratese") -> str:
    scores = {
        "Summarization": eval_results.SummarizationScore,
        "Coherence": eval_results.CoherenceScore,
        "Tonality": eval_results.TonalityScore,
        "Safety": eval_results.SafetyScore,
    }
    improvement_focus = [k for k, v in sorted(scores.items(), key=lambda x: x[1]) if v < 0.85]

    instructions = []
    if "Summarization" in improvement_focus:
        instructions += [
            "Improve coverage of core thesis and key supporting points.",
            "Ensure concise phrasing while preserving nuance.",
            "Mention feedback analysis and values alignment if missing.",
        ]
    if "Coherence" in improvement_focus:
        instructions += [
            "Reorder content for logical progression: overview -> key points -> closing.",
            "Use explicit transitions between sections.",
        ]
    if "Tonality" in improvement_focus:
        instructions += [
            f"Reinforce the requested tone: {tone}.",
            "Use formal institutional language and impersonal voice.",
            "Maintain consistent register throughout.",
        ]
    if "Safety" in improvement_focus:
        instructions += [
            "Ensure neutral, objective language.",
            "Avoid controversial or sensitive claims.",
            "Remove any potentially biased phrasing.",
        ]
    if not instructions:
        instructions = [
            "Maintain current quality while subtly enhancing clarity and tone consistency."
        ]

    prompt = (
        "You will improve the following summary in the requested style.\n\n"
        f"REQUESTED TONE: {tone}\n\n"
        "ORIGINAL SUMMARY:\n"
        + original_summary
        + "\n\nCURRENT EVALUATION:\n"
        + f"- Summarization ({eval_results.SummarizationScore:.2f}): {eval_results.SummarizationReason}\n"
        + f"- Coherence ({eval_results.CoherenceScore:.2f}): {eval_results.CoherenceReason}\n"
        + f"- Tonality ({eval_results.TonalityScore:.2f}): {eval_results.TonalityReason}\n"
        + f"- Safety ({eval_results.SafetyScore:.2f}): {eval_results.SafetyReason}\n\n"
        + "IMPROVEMENT INSTRUCTIONS:\n- "
        + "\n- ".join(instructions)
        + "\n\n"
        + "Return ONLY valid JSON:\n{\n  \"summary\": \"enhanced summary in the requested tone\",\n  \"improvements\": [\"bullet list of concrete changes you made\"]\n}\n"
    )
    return prompt


enhancement_prompt = build_enhancement_prompt(
    analysis.Summary, evaluation_results, tone=tone
)

enh_response = client.chat.completions.create(
    model=model_name,
    response_format={"type": "json_object"},
    temperature=0.3,
    messages=[
        {
            "role": "system",
            "content": f"You are a professional editor specializing in {tone} style. Improve summaries following instructions.",
        },
        {"role": "user", "content": enhancement_prompt},
    ],
)

enh_data = json.loads(enh_response.choices[0].message.content)
enhanced_summary = enh_data.get("summary", analysis.Summary)
improvements = enh_data.get("improvements", [])
if isinstance(improvements, str):
    improvements = [improvements]

# Basic sanity check – avoid degenerate very short summaries
if len(enhanced_summary.split()) < 50:
    enhanced_summary = analysis.Summary
    improvements.append("Generated summary was too short; retained original.")

print("=== ENHANCED SUMMARY ===\n")
print(enhanced_summary)

print("\n=== IMPROVEMENTS MADE ===")
for item in improvements:
    print("-", item)

# Re-evaluate enhanced summary
summarization_metric_enh = SummarizationMetric(
    model=model_name,
    assessment_questions=summarization_questions,
    threshold=0.0,
)
coherence_metric_enh = GEval(
    name="Coherence",
    evaluation_steps=coherence_questions,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model_name,
    threshold=0.0,
    strict_mode=False,
)
tonality_metric_enh = GEval(
    name="Tonality",
    evaluation_steps=tonality_questions,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model_name,
    threshold=0.0,
    strict_mode=False,
)
safety_metric_enh = GEval(
    name="Safety",
    evaluation_steps=safety_questions,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model_name,
    threshold=0.0,
    strict_mode=False,
)

enh_test_case = LLMTestCase(input=document_text, actual_output=enhanced_summary)

summarization_metric_enh.measure(enh_test_case)
coherence_metric_enh.measure(enh_test_case)
tonality_metric_enh.measure(enh_test_case)
safety_metric_enh.measure(enh_test_case)

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return 0.0

print("\n=== EVALUATION COMPARISON ===\n")
pairs = [
    (
        "Summarization",
        evaluation_results.SummarizationScore,
        safe_float(summarization_metric_enh.score),
    ),
    (
        "Coherence",
        evaluation_results.CoherenceScore,
        safe_float(coherence_metric_enh.score),
    ),
    (
        "Tonality",
        evaluation_results.TonalityScore,
        safe_float(tonality_metric_enh.score),
    ),
    (
        "Safety",
        evaluation_results.SafetyScore,
        safe_float(safety_metric_enh.score),
    ),
]
print("Metric        Original → Enhanced    Change")
print("-" * 45)
for name, orig, enh in pairs:
    change = enh - orig
    arrow = "↑" if change > 0 else "↓" if change < 0 else "→"
    print(f"{name:13} {orig:.2f} {arrow} {enh:.2f}    {change:+.2f}")


Output()

=== ENHANCED SUMMARY ===

In the distinguished treatise 'Managing Oneself' authored by Peter F. Drucker, the critical necessity of self-management within the modern knowledge economy is thoroughly examined. The document posits that individuals are required to function as their own chief executive officers, necessitating a profound comprehension of personal strengths, weaknesses, values, and work preferences. The utilization of feedback analysis is advocated as a strategic method to discern one's strengths and areas necessitating enhancement, with a pronounced emphasis on capitalizing on strengths rather than attempting to amend weaknesses. Furthermore, the treatise delves into the importance of recognizing one's preferred learning modality, whether as a reader or listener, and the subsequent adaptation of work habits to optimize efficiency. It underscores the imperative of aligning personal values with those of the organization to preclude frustration and underperformance. Additionally

Output()

Output()

Output()


=== EVALUATION COMPARISON ===

Metric        Original → Enhanced    Change
---------------------------------------------
Summarization 0.73 ↑ 0.75    +0.02
Coherence     0.91 ↓ 0.90    -0.01
Tonality      0.90 ↑ 0.90    +0.00
Safety        1.00 → 1.00    +0.00


Please, do not forget to add your comments.

# My Comments



### Enhancement Results Analysis

* **Summarization:** Improved (0.73 $\rightarrow$ 0.75, +0.02).
    * **Reason:** This gain likely came from clarifying the role of feedback analysis and value alignment, as noted in your improvements list.
* **Coherence:** Decreased slightly (0.91 $\rightarrow$ 0.90, -0.01).
    * **Reason:** This minimal drop is likely due to the highly formal, passive, and dense "Bureaucratese" style, which can make text slightly harder to parse.
* **Tonality:** Unchanged (0.90 $\rightarrow$ 0.90, +0.00).
    * **Reason:** The score was already high and successfully locked into the target formal tone.
* **Safety:** Unchanged (1.00).

### Conclusion

This was a successful revision. I improved summarization quality while perfectly maintaining the desired formal tone. The tiny 0.01 drop in coherence is a negligible trade-off for the stylistic requirements. The scores are now stable and high.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
